In [2]:
import pandas as pd
import numpy as np

## Verify features-tpase tags match FASTA headers

Check that the sequence IDs in features-tpase exist in all_vgp_tes.fa and that the tags are consistent.

In [3]:
# Load FASTA headers
def get_fasta_headers(file_path):
    """Extract all headers from FASTA file"""
    headers = {}
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('>'):
                header = line.strip()
                # Extract the class from header (after #)
                if '#' in header:
                    te_class = header.split('#')[1]
                else:
                    te_class = None
                headers[header] = te_class
    return headers

# Load features-tpase file
def load_features_tpase(file_path):
    """Load features-tpase file: header -> tag mapping"""
    features = {}
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                header, tag = parts[0], parts[1]
                features[header] = tag
    return features

# Load both files
fasta_path = '../data/vgp/all_vgp_tes.fa'
features_path = '../data/vgp/features-tpase'

fasta_headers = get_fasta_headers(fasta_path)
features_tpase = load_features_tpase(features_path)

print(f"FASTA file has {len(fasta_headers)} sequences")
print(f"Features-tpase file has {len(features_tpase)} entries")

FASTA file has 135751 sequences
Features-tpase file has 135751 entries


In [4]:
# Check 1: Do all features-tpase headers exist in FASTA?
missing_in_fasta = [h for h in features_tpase.keys() if h not in fasta_headers]
print(f"Headers in features-tpase but NOT in FASTA: {len(missing_in_fasta)}")
if missing_in_fasta:
    print("Examples of missing headers:")
    for h in missing_in_fasta[:5]:
        print(f"  {h}")

# Check 2: Do the tags match the class in FASTA header?
mismatches = []
matches = 0
for header, tag in features_tpase.items():
    if header in fasta_headers:
        fasta_class = fasta_headers[header]
        if fasta_class != tag:
            mismatches.append((header, tag, fasta_class))
        else:
            matches += 1

print(f"\nTag matches: {matches}")
print(f"Tag mismatches: {len(mismatches)}")
if mismatches:
    print("\nFirst 10 mismatches (header, features-tag, fasta-class):")
    for h, tag, fasta_class in mismatches[:10]:
        print(f"  {h}")
        print(f"    features-tpase tag: {tag}")
        print(f"    FASTA class:        {fasta_class}")

Headers in features-tpase but NOT in FASTA: 0

Tag matches: 10650
Tag mismatches: 125101

First 10 mismatches (header, features-tag, fasta-class):
  >ERV1_15-aAnoBae#LTR/ERV1
    features-tpase tag: None
    FASTA class:        LTR/ERV1
  >ERV1_1-aAnoBae#LTR/ERV1
    features-tpase tag: None
    FASTA class:        LTR/ERV1
  >ERV1_2-aAnoBae#LTR/ERV1
    features-tpase tag: None
    FASTA class:        LTR/ERV1
  >ERV1_3-aAnoBae#LTR/ERV1
    features-tpase tag: None
    FASTA class:        LTR/ERV1
  >Gypsy_478-aAnoBae#LTR/Gypsy
    features-tpase tag: None
    FASTA class:        LTR/Gypsy
  >Gypsy_488-aAnoBae#LTR/Gypsy
    features-tpase tag: None
    FASTA class:        LTR/Gypsy
  >Gypsy_494-aAnoBae#LTR/Gypsy
    features-tpase tag: None
    FASTA class:        LTR/Gypsy
  >Gypsy_498-aAnoBae#LTR/Gypsy
    features-tpase tag: None
    FASTA class:        LTR/Gypsy
  >Gypsy_500-aAnoBae#LTR/Gypsy
    features-tpase tag: None
    FASTA class:        LTR/Gypsy
  >Gypsy_501-aAnoBae#LTR/G

In [5]:
# Summary statistics of tags in features-tpase
from collections import Counter

tag_counts = Counter(features_tpase.values())
print("Tag distribution in features-tpase:")
print(f"Total unique tags: {len(tag_counts)}\n")
for tag, count in tag_counts.most_common():
    pct = 100 * count / len(features_tpase)
    print(f"  {tag}: {count} ({pct:.2f}%)")

Tag distribution in features-tpase:
Total unique tags: 18

  None: 125101 (92.15%)
  DNA/hAT: 4900 (3.61%)
  DNA/TcMar-Tc1: 1875 (1.38%)
  DNA: 1530 (1.13%)
  DNA/PIF-Harbinger: 790 (0.58%)
  DNA/PiggyBac: 563 (0.41%)
  DNA/Academ-1: 415 (0.31%)
  DNA/CMC: 180 (0.13%)
  DNA/Sola-2: 93 (0.07%)
  DNA/Kolobok: 86 (0.06%)
  DNA/P: 77 (0.06%)
  DNA/Sola-1: 43 (0.03%)
  DNA/PIF-ISL2EU: 38 (0.03%)
  DNA/MULE-MuDR: 23 (0.02%)
  DNA/Crypton-V: 19 (0.01%)
  DNA/Merlin: 12 (0.01%)
  DNA/Ginger-1: 4 (0.00%)
  DNA/Dada: 2 (0.00%)


In [6]:
def load_fa(file_dir):
    with open(file_dir, 'r') as f:
        lines = f.readlines()
        
    te_dict = {}
    for l in lines:
        if l.startswith('>'):
            key = l[1:-1]
            ind = len(key.split('_')[0]) + key.split('_')[1].index('-') + 1
            id = key[:ind]
            tag = key[ind + 1: key.index('#')]
            te_class = key[key.index('#') + 1:]
        else:
            seq = l[:-1]
            te_dict[key] = {'key': key, 'id': id, 'tag': tag, 'class': te_class, 'seq': seq}
    
    return te_dict

In [7]:
vgp_te_dict = load_fa('../data/vgp/all_vgp_tes.fa')

In [8]:
with open('../data/vgp/features.txt', 'r') as f:
    lines = f.readlines()
    
    for l in lines:
        key, feature = l[1:-1].split('\t')
        if feature == "TRUE":
            vgp_te_dict[key]['feature'] = 1
        else:
            vgp_te_dict[key]['feature'] = 0

FileNotFoundError: [Errno 2] No such file or directory: '../data/vgp/features.txt'

In [ ]:
vgp_te_df = pd.DataFrame.from_dict(vgp_te_dict, orient='index', columns=['id', 'tag', 'class', 'seq', 'feature'])

In [ ]:
vgp_te_df.iloc[:10]

,id,tag,class,seq,feature
hAT_1-aAnoBae#DNA/hAT,hAT_1,aAnoBae,DNA/hAT,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,1
hAT_131-aAnoBae#DNA/hAT,hAT_131,aAnoBae,DNA/hAT,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,1
hAT_147-aAnoBae#DNA/hAT,hAT_147,aAnoBae,DNA/hAT,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,0
hAT_210-aAnoBae#DNA/hAT,hAT_210,aAnoBae,DNA/hAT,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,0
hAT_254-aAnoBae#DNA/hAT,hAT_254,aAnoBae,DNA/hAT,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,0
ERV1_15-aAnoBae#LTR/ERV1,ERV1_15,aAnoBae,LTR/ERV1,TGTTGAGGATAAAGAATTGAGTATCCAAAAATACTTAATTCAGCCA...,0
hAT_292-aAnoBae#DNA/hAT,hAT_292,aAnoBae,DNA/hAT,TAGAGATGAGCGAACCGGTCCCGGTTCGGCTCGAGGCGGTTCGCCG...,0
hAT_298-aAnoBae#DNA/hAT,hAT_298,aAnoBae,DNA/hAT,TAGAGATGAGCGAACCGGTCCCGGTTCGGCTCGAGGCGGTTCGCCG...,0
hAT_299-aAnoBae#DNA/hAT,hAT_299,aAnoBae,DNA/hAT,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,0
PIF-Harbinger_1-aAnoBae#DNA/PIF-Harbinger,PIF-Harbinger_1,aAnoBae,DNA/PIF-Harbinger,AGGGGTGCTTCACACATAGCGAGATCGCTAGCGAGATCGCTGCTGA...,0


In [ ]:
pd.to_pickle(vgp_te_df, '../data/vgp/vgp_te_features.pkl')

In [ ]:
vgp_features = vgp_te_df[['seq', 'feature']].reset_index(drop=True)
vgp_features

,seq,feature
0,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,1
1,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,1
2,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,0
3,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,0
4,TAGAGTTGAGCGCGGTTCGTGGTTCGTGGTTCTCCAGTTCGCGGCT...,0
...,...,...
135746,GTTTTTGATCATAGATTTCATAGACTTTTTCATAGATTTTACAGTG...,0
135747,ATATTCAAAGACTTTTTTTTTTTAAATTTAGATTACCCAATTATTT...,0
135748,CTGGAATATCCATGATGTGAAGATGCCGGCGTTGGACTGGGGTGAG...,0
135749,CCCCACCTCCCAGGGGCTGGTTTAGCTCACNAGGGGCTGGTTTAGC...,0


In [ ]:
sum(vgp_features['feature']) / len(vgp_features)

0.2977804951713063

In [ ]:
len_list = np.array([len(vgp_features['seq'][i]) for i in range(len(vgp_features))])
len_list.min(), len_list.max(), len_list.mean(), np.median(len_list)

(173, 19907, 3791.0058710433073, 3263.0)

In [ ]:
ENCODE = np.full(256, 4, dtype=np.int64)
for ch, idx in zip(b"ACGTacgt", [0, 1, 2, 3, 0, 1, 2, 3]):
    ENCODE[ch] = idx
REV_COMP = np.array([3, 2, 1, 0, 4]) 

for i in range(10):
    seq = np.array(vgp_features['seq'])[i]
    seq_encoded = ENCODE[np.frombuffer(seq.encode(), dtype=np.uint8)]
    rev_comp = REV_COMP[seq_encoded][::-1]
    rc_seq = ''.join(np.array(['A', 'C', 'G', 'T', 'N'])[rev_comp])
    
    feature = vgp_features['feature'][i]
    print(feature)
    print(seq)
    print(rc_seq)

1
...
1
...
0
...
0
...
0
...
0
...
0
...
0
...
0
...
0
...


In [ ]:
seq

'AGGGGTGCTTCACACATAGCGAGATCGCTAGCGAGATCGCTGCTGAGTCACGGTTTTGGTGACGCAATAGTGACCTCATTAGCGATCTCGCTGTGTGTGACACTGAGCAGCGATCTGGCCCCTGCTGTGAGATCGCTGCTCGTTACACACAGTGCTGGTTCGTTTTCTTCAAAGGCGATGTCCTGCTGTGCAGGACGCAATTCTGTGTGTGACTCCTAGACAGCGACCTTGTTCACGACTNACGTAGGCGTGCATCTTCGTTGGTTTCCACGCCCTCTCTGTTCTGATTGGTGATCGCTACTGCGTTCTCATTGGACGTCTCATTGGGCGTCTTCCATTGTTTGGTAGGGCTAGTTCACTTGTCCCTGACCGCGTGTAGCAAGCAGATCGCAATACAGTCCATTCTGCATCGCTTCATCCGTTCATCAGATATTCTGGGCGTTGTACTCTGCAAGGTAGGAATTTTTTTGAGGTCGCAAATACATTTCCATAATATTATTACATTTCTATAAGTATCTATCATTGTGCTATAGAACATTGTGTGTTCATGCTTCTACACTTGCAAGCAAATGGTGAGAGGGCTTGGCTACTATCTGTACACATGCTTTTCTATGGGCAGACATTTTAACCCCTTCAAGACTGACCTTTTTTTTGCTTTACGTTTTTATTTCGCCATTCTTTTTCTGAGAGACGTAATTTTTTTATTTTACAGTCAATATGGTCATGTGAGGGCTCATTTTTTTGCGGAAATAACTGTGCTTTTAAATGAAACCATAAGTTTTCCCATATAGTGTACTGTAAACCGTCCAAAAATTCCAAATGAAGAAAAATTGTAAAATAAGTGCGATAGCGTCTCAATCACATGTTTTTACCGCTAACCATTTTTGCGCATATGATATGTTTGGCTCGCCTATTATTTCATTTAGCACAAAAGTTGTGATGACATAAAAATGTCATTTTGGCGTTTGAAATGTTTTTGCCGCTATGCCGTATCGTGAT

Highly accurate ab initio gene annotation with ANNEVO

BERT AlphaGenome

## tpase

In [ ]:
with open('../data/vgp/20251215-features-tpase', 'r') as f:
    lines = f.readlines()
    label_dict = {}
    for l in lines:
        header, label = l.strip().split('\t')
        label_dict[header] = label_dict.get(header, label)

label_mismatches, mismatches = {}, 0
for key in list(label_dict.keys()):
    trailing = key.split('#')[1]
    if trailing != label_dict[key] and ('DNA' in key or 'DNA' in trailing):
        label_mismatches[key] = (label_dict[key], trailing)
        mismatches += 1

print(f"Total mismatches: {mismatches}")
label_mismatches

Total mismatches: 18856


{'>Unknown_36-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_42-aAnoBae#DNA': ('None', 'DNA'),
 '>Unknown_47-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_48-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_51-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_52-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_53-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_57-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>hAT_111-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_59-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_60-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_61-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_62-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_66-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_70-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_72-aAnoBae#DNA/hAT': ('None', 'DNA/hAT'),
 '>Unknown_77-aAnoBae#DNA': ('None', 'DNA'),
 '>Unknown_78-aAnoBae#DNA': ('None', 'DNA'),
 '>hAT_112-aAnoBae#DNA': ('None', 'DNA'),
 '>Unknown_79-aAnoBae#DNA': ('N

In [10]:
def load_labels_binary(path):
    label_dict = {}
    with open(path, 'r') as f:
        for line in f:
            if not line:
                continue
            parts = line.strip().split('\t')
            header = parts[0][1:]
            if parts[1] == "None":
                label = 0.0
            else:
                label = 1.0
            label_dict[header] = label
    return label_dict

label_dict = load_labels_binary("../data/vgp/20251215-features-tpase")

In [11]:
label_dict

{'hAT_1-aAnoBae#DNA/hAT': 1.0,
 'hAT_131-aAnoBae#DNA/hAT': 1.0,
 'hAT_147-aAnoBae#DNA/hAT': 1.0,
 'hAT_210-aAnoBae#DNA/hAT': 1.0,
 'hAT_254-aAnoBae#DNA/hAT': 1.0,
 'ERV1_15-aAnoBae#LTR/ERV1': 0.0,
 'hAT_292-aAnoBae#DNA/hAT': 1.0,
 'hAT_298-aAnoBae#DNA/hAT': 1.0,
 'hAT_299-aAnoBae#DNA/hAT': 1.0,
 'PIF-Harbinger_1-aAnoBae#DNA/PIF-Harbinger': 1.0,
 'hAT_23-aAnoBae#DNA/hAT': 1.0,
 'ERV1_1-aAnoBae#LTR/ERV1': 0.0,
 'hAT_39-aAnoBae#DNA/hAT': 1.0,
 'hAT_69-aAnoBae#DNA/hAT': 1.0,
 'hAT_90-aAnoBae#DNA/hAT': 1.0,
 'hAT_110-aAnoBae#DNA/hAT': 1.0,
 'hAT_116-aAnoBae#DNA/hAT': 1.0,
 'hAT_124-aAnoBae#DNA/hAT': 1.0,
 'hAT_130-aAnoBae#DNA/hAT': 1.0,
 'ERV1_2-aAnoBae#LTR/ERV1': 0.0,
 'ERV1_3-aAnoBae#LTR/ERV1': 0.0,
 'hAT_132-aAnoBae#DNA/hAT': 1.0,
 'hAT_133-aAnoBae#DNA/hAT': 1.0,
 'hAT_134-aAnoBae#DNA/hAT': 1.0,
 'hAT_136-aAnoBae#DNA/hAT': 1.0,
 'hAT_137-aAnoBae#DNA/hAT': 1.0,
 'hAT_138-aAnoBae#DNA/hAT': 1.0,
 'Gypsy_478-aAnoBae#LTR/Gypsy': 0.0,
 'Gypsy_488-aAnoBae#LTR/Gypsy': 0.0,
 'hAT_148-aAnoBae#DNA/

In [ ]:
def load_labels(path):
    label_dict = {}
    labels, index = {}, 0
    with open(path, 'r') as f:
        for line in f:
            if not line:
                continue
            parts = line.strip().split('\t')
            header = parts[0][1:]
            
            if parts[1] not in labels.keys():
                labels[parts[1]] = index
                index += 1
            label_dict[header] = labels[parts[1]]
            
    return label_dict

label_dict = load_labels("../data/vgp/features-tpase.txt")

In [ ]:
label_dict

{'hAT_1-aAnoBae#DNA/hAT': 0,
 'hAT_131-aAnoBae#DNA/hAT': 0,
 'hAT_147-aAnoBae#DNA/hAT': 0,
 'hAT_210-aAnoBae#DNA/hAT': 0,
 'hAT_254-aAnoBae#DNA/hAT': 0,
 'ERV1_15-aAnoBae#LTR/ERV1': 1,
 'hAT_292-aAnoBae#DNA/hAT': 0,
 'hAT_298-aAnoBae#DNA/hAT': 0,
 'hAT_299-aAnoBae#DNA/hAT': 0,
 'PIF-Harbinger_1-aAnoBae#DNA/PIF-Harbinger': 2,
 'hAT_23-aAnoBae#DNA/hAT': 0,
 'ERV1_1-aAnoBae#LTR/ERV1': 1,
 'hAT_39-aAnoBae#DNA/hAT': 0,
 'hAT_69-aAnoBae#DNA/hAT': 0,
 'hAT_90-aAnoBae#DNA/hAT': 0,
 'hAT_110-aAnoBae#DNA/hAT': 0,
 'hAT_116-aAnoBae#DNA/hAT': 0,
 'hAT_124-aAnoBae#DNA/hAT': 0,
 'hAT_130-aAnoBae#DNA/hAT': 0,
 'ERV1_2-aAnoBae#LTR/ERV1': 1,
 'ERV1_3-aAnoBae#LTR/ERV1': 1,
 'hAT_132-aAnoBae#DNA/hAT': 0,
 'hAT_133-aAnoBae#DNA/hAT': 0,
 'hAT_134-aAnoBae#DNA/hAT': 0,
 'hAT_136-aAnoBae#DNA/hAT': 0,
 'hAT_137-aAnoBae#DNA/hAT': 0,
 'hAT_138-aAnoBae#DNA/hAT': 0,
 'Gypsy_478-aAnoBae#LTR/Gypsy': 1,
 'Gypsy_488-aAnoBae#LTR/Gypsy': 1,
 'hAT_148-aAnoBae#DNA/hAT': 0,
 'Gypsy_494-aAnoBae#LTR/Gypsy': 1,
 'Gypsy_498-aA

In [ ]:
count = [0 for _ in range(18)]
for key in label_dict.keys():
    count[label_dict[key]] += 1
count

[4900,
 125101,
 790,
 563,
 180,
 1875,
 1530,
 23,
 93,
 4,
 12,
 415,
 43,
 38,
 19,
 86,
 77,
 2]